In [1]:
# importing the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import os
import joblib
from sklearn.preprocessing import MinMaxScaler

In [2]:
# loading the datasets

telco = pd.read_csv("../data/Telco-Customer-Churn.csv")
subscription = pd.read_csv("../data/Subscription_Service_Churn_Dataset.csv")
ecom = pd.read_csv("../data/ecommerce_transactions.csv")

In [3]:
# Standardizing the column names

telco.columns = telco.columns.str.lower().str.strip()
subscription.columns = subscription.columns.str.lower().str.strip()
ecom.columns = ecom.columns.str.lower().str.strip()

In [4]:
# Editing column names to ease the process of merging

telco.rename(columns={'customerid' : 'customer_id'}, inplace=True)

#subscription.rename(columns={'customerid' : 'customer_id'}, inplace=True)
subscription.rename(columns={'payment_method' : 'paymentmethod'}, inplace=True)
subscription.rename(columns={'subscription_length' : 'tenure'}, inplace=True)
subscription.rename(columns={'monthly_spend' : 'monthlycharges'}, inplace=True)
subscription.rename(columns={'churned' : 'churn'}, inplace=True)

ecom.rename(columns={'payment_method' : 'paymentmethod'}, inplace=True)

In [5]:
# Creating customer_id column in the e-commerce dataset

# Adding prefix to make customer_IDs unique
telco['customer_id'] = telco['customer_id'].apply(lambda x: f"TEL_{x}")
#subscription['customer_id'] = subscription['customer_id'].apply(lambda x: f"SUBS_{x}")

# Creating customer_id in e-commerce dataset
if 'user_name' in ecom.columns:
    user_id_map = {name: f"ECOM_{idx+1}" for idx, name in enumerate(ecom['user_name'].unique())}
    ecom['customer_id'] = ecom['user_name'].map(user_id_map)

# Creating customer_id in subscription dataset
if 'customer_id' in subscription.columns:
    user_id_map = {name: f"SUBS_{idx+1}" for idx, name in enumerate(subscription['customer_id'].unique())}
    subscription['customer_id'] = subscription['customer_id'].map(user_id_map)


In [6]:
#creating churn column in ecommerce dataset

mask_ecom = ecom['customer_id'].str.startswith('ECOM_')
options = [0, 1]
ecom.loc[mask_ecom, 'churn'] = np.random.choice(options, size=mask_ecom.sum())

In [7]:
#undersampling ecommerce dataset using stratification

def stratified_undersample(df, target_size, stratify_cols, random_state=42):
    df = df.copy()
    
    # Create stratification key
    df["_strat_key"] = df[stratify_cols].astype(str).agg("_".join, axis=1)

    total = len(df)
    frac = target_size / total

    parts = []

    # Stratified proportional sampling
    for key, group in df.groupby("_strat_key"):
        n = max(1, int(len(group) * frac))
        parts.append(group.sample(n=n, random_state=random_state))

    df_partial = pd.concat(parts, ignore_index=True)

    # If partial sample is smaller than target → sample extra rows
    if len(df_partial) < target_size:
        needed = target_size - len(df_partial)
        extra = df.sample(n=needed, random_state=random_state)
        df_partial = pd.concat([df_partial, extra], ignore_index=True)

    # If partial sample is larger → trim
    df_final = df_partial.sample(n=target_size, random_state=random_state)

    return df_final.drop(columns=["_strat_key"]).reset_index(drop=True)

target_size = 7043
strat_cols = ["churn", "paymentmethod"]
df_ecom_7k = stratified_undersample(ecom, target_size, strat_cols)

print(df_ecom_7k.shape)
print(df_ecom_7k["churn"].value_counts())
print(df_ecom_7k["paymentmethod"].value_counts())
#print(df_ecom_7k["Gender"].value_counts())

(7043, 10)
churn
1.0    3537
0.0    3506
Name: count, dtype: int64
paymentmethod
UPI                 1193
Cash on Delivery    1188
Debit Card          1178
Credit Card         1171
PayPal              1162
Net Banking         1151
Name: count, dtype: int64


In [8]:
#oversampling subscription dataset

def oversample_to_target(df, target_size, random_state=42, noise_std=1e-6):
    
    df = df.copy().reset_index(drop=True)
    current_size = len(df)

    # If already big enough → simply downsample
    if current_size >= target_size:
        return df.sample(n=target_size, random_state=random_state).reset_index(drop=True)

    needed = target_size - current_size

    # Identify numeric columns for noise addition
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Oversample using bootstrapping
    df_extra = df.sample(n=needed, replace=True, random_state=random_state).copy()

    # Add tiny noise ONLY to numeric columns to avoid duplicates
    for col in numeric_cols:
        df_extra[col] += np.random.normal(0, noise_std, size=len(df_extra))

    # Combine original + new oversampled rows
    df_final = pd.concat([df, df_extra], ignore_index=True)

    return df_final.reset_index(drop=True)


target_size = 7043  # your required size
df_subs_7043 = oversample_to_target(subscription, target_size)

print(df_subs_7043.shape)

(7043, 12)


In [9]:
# Combining the datasets

#finding all the columns in the dataset
all_cols = set(telco.columns) | set(df_subs_7043.columns) | set(df_ecom_7k.columns)

#adding missing columns to the datasets
for df in [telco, df_subs_7043, df_ecom_7k]:
    for col in all_cols:
        if col not in df.columns:
            df[col] = None

#Reordering the columns in a similar order in all datasets
telco = telco[sorted(all_cols)]
df_subs_7043 = df_subs_7043[sorted(all_cols)]
df_ecom_7k = df_ecom_7k[sorted(all_cols)]

#Combining the datasets (row-wise)
combined_df = pd.concat([telco, df_subs_7043, df_ecom_7k], ignore_index=True)

C:\Users\KARISHMA\AppData\Local\Temp\ipykernel_24908\663097727.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([telco, df_subs_7043, df_ecom_7k], ignore_index=True)


In [12]:
# checking if missing values are present

combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21129 entries, 0 to 21128
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   age                     13371 non-null  float64
 1   churn                   21129 non-null  object 
 2   contract                7043 non-null   object 
 3   country                 7043 non-null   object 
 4   customer_id             21129 non-null  object 
 5   dependents              7043 non-null   object 
 6   deviceprotection        7043 non-null   object 
 7   discount_offered        7043 non-null   float64
 8   gender                  14086 non-null  object 
 9   internetservice         7043 non-null   object 
 10  last_activity           7043 non-null   float64
 11  monthlycharges          14086 non-null  float64
 12  multiplelines           7043 non-null   object 
 13  onlinebackup            7043 non-null   object 
 14  onlinesecurity          7043 non-null 

In [13]:
# Saving intermediate cleaned dataset

combined_df.to_csv("encoded_churn_dataset_2.csv", index=False)
print("Dataset saved successfully")

Dataset saved successfully


In [14]:
#filling in the dataset with gender

def assign_gender(name):
    name = str(name).split()[0].lower()
    if name.endswith(('a', 'e', 'ie')):
        return 'Female'
    else:
        return 'Male'

ecom_mask = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'gender'] = combined_df.loc[ecom_mask,'user_name'].apply(assign_gender)

In [15]:
# filling in the tenure column for customer_ids starting with 'ECOM_'

ecom_mask = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'tenure'] = np.random.randint(1, 70, size=ecom_mask.sum())

#sub_mask = combined_df['customer_id'].str.startswith('SUBS_')
#combined_df.loc[sub_mask, 'tenure'] = np.random.randint(1, 70, size=sub_mask.sum())

In [16]:
# filling in random monthlycharges value for customers with id 'ECOM_'

mask_ecom = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'monthlycharges'] = np.round(np.random.uniform(20, 120, size=mask_ecom.sum()), 2)

#mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
#missing_subs = mask_subs & combined_df['monthlycharges'].isna()
#combined_df.loc[missing_subs, 'monthlycharges'] = np.round(np.random.uniform(20, 120, size=missing_subs.sum()), 2)

In [17]:
#editing the churn column

#ecom doesn't have churn so randomly assigning the values
#mask_ecom = combined_df['customer_id'].str.startswith('ECOM_')
#options = ['Yes', 'No']
#combined_df.loc[mask_ecom, 'churn'] = np.random.choice(options, size=mask_ecom.sum())

#mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
#combined_df.loc[mask_subs, 'churn'] = combined_df.loc[mask_subs, 'churn'].replace({1: 'Yes', 0: 'No'})


#converting churn values of customer_id 'SUB_S' to a proper value
mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
combined_df.loc[mask_subs, 'churn'] = (combined_df.loc[mask_subs, 'churn'].astype(float) > 0.5).astype(int)

#converting yes to 1 and no to 0 for churn column for customer's whose id starts with 'TEL_'
mask_tel = combined_df['customer_id'].str.startswith('TEL_')
combined_df.loc[mask_tel, 'churn'] = (combined_df.loc[mask_tel, 'churn'].replace({'Yes' : 1, 'No' : 0, 'yes' : 1, 'no' : 0}).astype(int))


C:\Users\KARISHMA\AppData\Local\Temp\ipykernel_24908\23771064.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined_df.loc[mask_tel, 'churn'] = (combined_df.loc[mask_tel, 'churn'].replace({'Yes' : 1, 'No' : 0, 'yes' : 1, 'no' : 0}).astype(int))


In [18]:
#editing tenure column to remove decimal values
mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
combined_df.loc[mask_subs, 'tenure'] = combined_df.loc[mask_subs, 'tenure'].round().astype(int)

In [19]:
#converting the datatypes of age and tenure to int

combined_df['age'] = combined_df['age'].astype(int)
combined_df['tenure'] = combined_df['tenure'].astype(int)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer